In [1]:
import pandas as pd
import numpy as np
import sys
from tqdm.auto import tqdm
from antropy import perm_entropy
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneGroupOut

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, lowpass_filter_causal, bootstrap_report

/home/marco/git/online-impairment-detection/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sampling_rate = 100 # Hz, sensor sampling frequency
lpf_cutoff = 10 # Hz, FIR lowpass cutoff
lpf_order = 64 # FIR filter taps
pe_order = 5 # permutation entropy embedding dimension
pe_delay = int(sampling_rate / (2 * lpf_cutoff)) # PE delay (Nyquist-matched)
pe_w = 1000 # entropy window size in samples
pe_s = 100 # window step size in samples
alpha = 0.023 # false-alarm bound
glue = False # set True to glue metrics for result collection
per_ride_path = None # set a path to save per-ride outcomes as CSV
J = 0.01 # Simple Jumper jump probability
seed = 0 # RNG seed for randomized conformal p-value smoothing
rng = np.random.default_rng(seed)

In [3]:
# Parameters
alpha = 0.023
glue = True
per_ride_path = None


In [4]:
data = pd.read_parquet("../data/data.parquet")
data

,ride_id,time_index,participant_id,run,trial,ax,ay,az,rx,ry,rz,throttle,brake_l,brake_r
0,000e5ab725f4,0,23.0,Sober,2.0,-0.442073,-9.215956,-3.105285,0.020311,-0.136070,0.213650,41.0,41.0,41.0
1,000e5ab725f4,10,23.0,Sober,2.0,-0.547358,-9.390632,-2.988635,0.019853,-0.149967,0.210443,41.0,41.0,41.0
2,000e5ab725f4,20,23.0,Sober,2.0,-0.552742,-9.348757,-2.974876,0.017715,-0.165086,0.208457,41.0,41.0,41.0
3,000e5ab725f4,30,23.0,Sober,2.0,-0.605384,-9.287740,-3.065803,0.014508,-0.182190,0.206319,41.0,41.0,41.0
4,000e5ab725f4,40,23.0,Sober,2.0,-0.631106,-9.226125,-3.071786,0.014966,-0.197767,0.204181,41.0,41.0,41.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
876191,fdc39ec37c87,67410,13.0,Sober,2.0,1.475773,-10.374680,-0.763908,-0.276263,1.067486,0.098043,147.0,41.0,41.0
876192,fdc39ec37c87,67420,13.0,Sober,2.0,1.175474,-9.505487,-1.128814,-0.290924,1.067792,0.093462,147.0,41.0,41.0
876193,fdc39ec37c87,67430,13.0,Sober,2.0,1.273579,-10.332806,-0.802791,-0.313373,1.102611,0.105374,147.0,41.0,41.0
876194,fdc39ec37c87,67440,13.0,Sober,2.0,1.231107,-9.077770,-1.895115,-0.332616,1.102306,0.106443,147.0,41.0,41.0


In [5]:
labels = data.groupby(["ride_id", "participant_id"])["run"].first().reset_index()
labels

,ride_id,participant_id,run
0,000e5ab725f4,23.0,Sober
1,02a49cf28afc,31.0,High
2,04a58ca54c11,28.0,Sober
3,065ea8ad5f24,23.0,Low
4,097169d2138f,10.0,Low
...,...,...,...
136,fbf0250f1611,8.0,High
137,fc09a596ffcd,25.0,High
138,fcb8375a05a2,24.0,Low
139,fd0f1af171a2,23.0,Low


In [6]:
features = ["ax", "ay", "az", "rx", "ry", "rz", "throttle"]

data_np = reshape_to_numpy(
    data,
    features = features,
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_lpf = lowpass_filter_causal(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=lpf_cutoff,
    gyro_cutoff=lpf_cutoff,
    order=lpf_order,
    fs=sampling_rate,
)

In [7]:
# Precompute windowed entropies for all rides.
n_channels = len(features)
all_windowed_ent = []

for ride in tqdm(data_np_lpf, desc="windowed entropy"):
    # get row indices without any NaN
    valid = np.flatnonzero(~np.any(np.isnan(ride), axis=1))
    # strip leading (FIR warm-up) and trailing NaNs
    ride_trimmed = ride[valid[0]:valid[-1] + 1]
    # sliding-window permutation entropy per channel
    n_steps = max(0, (len(ride_trimmed) - pe_w) // pe_s + 1)
    ride_ent = np.empty((n_steps, n_channels))
    for t in range(n_steps):
        window = ride_trimmed[t * pe_s : t * pe_s + pe_w]
        for j in range(n_channels):
            ride_ent[t, j] = perm_entropy(window[:, j],
                                          order=pe_order, delay=pe_delay, normalize=True)
    all_windowed_ent.append(ride_ent)

windowed entropy:   0%|                                                                                                                                                         | 0/141 [00:00<?, ?it/s]

windowed entropy:   1%|██                                                                                                                                               | 2/141 [00:00<00:08, 16.55it/s]

windowed entropy:   3%|████                                                                                                                                             | 4/141 [00:00<00:09, 15.03it/s]

windowed entropy:   4%|██████▏                                                                                                                                          | 6/141 [00:00<00:09, 14.36it/s]

windowed entropy:   6%|████████▏                                                                                                                                        | 8/141 [00:00<00:08, 15.45it/s]

...

windowed entropy:  96%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉      | 135/141 [00:08<00:00, 15.99it/s]

windowed entropy:  97%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉    | 137/141 [00:08<00:00, 16.77it/s]

windowed entropy:  99%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉  | 139/141 [00:08<00:00, 16.46it/s]

windowed entropy: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 141/141 [00:08<00:00, 15.99it/s]

windowed entropy: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 141/141 [00:08<00:00, 16.06it/s]

In [8]:
# Within-subject centering (WSC): subtract per-channel mean of the
# participant's other rides to remove between-subject variability
# without needing to know which ride is sober.
pids = labels["participant_id"].values
all_windowed_wsc = []

for i in range(len(labels)): # i is a positional index, not a label
    # indices of the same participant's other rides
    same_pid = np.flatnonzero(pids == pids[i])
    other_idx = same_pid[same_pid != i]
    # collect their entropy windows
    other_windows = [all_windowed_ent[j] for j in other_idx]
    # per-channel mean across those rides
    mu = np.nanmean(np.vstack(other_windows), axis=0)
    # subtract to center within subject
    all_windowed_wsc.append(all_windowed_ent[i] - mu)

In [9]:
# LOPO CV
logo = LeaveOneGroupOut()
results = []

for train_idx, test_idx in tqdm(list(logo.split(labels, labels["run"], labels["participant_id"])), desc="LOPO"):
    # cross-conformal calibration via inner LOPO
    train_labels = labels.iloc[train_idx].reset_index(drop=True)
    inner_logo = LeaveOneGroupOut()
    folds = [] # list of (clf, fold_cal_scores) tuples

    for inn_tr, inn_te in inner_logo.split(train_labels, train_labels["run"], train_labels["participant_id"]):
        # split inner-training windows by label
        inn_sober = [all_windowed_wsc[train_idx[j]] for j in inn_tr if train_labels["run"].iloc[j] == "Sober"]
        inn_imp   = [all_windowed_wsc[train_idx[j]] for j in inn_tr if train_labels["run"].iloc[j] != "Sober"]

        # fit logistic regression; P(impaired | x) as non-conformity score
        X_in = np.vstack(inn_sober + inn_imp)
        y_in = np.concatenate([np.zeros(sum(map(len, inn_sober))), np.ones(sum(map(len, inn_imp)))])
        clf = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_in, y_in)

        # build calibration set (held-out sober rides only)
        fold_cal = []
        for j in inn_te:
            if train_labels["run"].iloc[j] == "Sober":
                x = all_windowed_wsc[train_idx[j]]
                fold_cal.extend(clf.predict_proba(x)[:, 1])
        folds.append((clf, np.array(fold_cal)))

    # compute log-wealth path for held-out participant
    for ride_idx in test_idx:
        # load test ride windows
        ride_wsc = all_windowed_wsc[ride_idx]
        n_windows = len(ride_wsc)

        # init Simple Jumper state
        C = {-1: 1/3, 0: 1/3, 1: 1/3} # split across the three betting regimes {-1, 0, 1}
        log_wealth_path = [] # log-wealth log(S_n) at each step
        n_ties = 0 # number of p-values where score tied a calibration score

        # sequential betting with cross-conformal p-values
        for t in range(n_windows):
            # get current window
            x = ride_wsc[t:t+1]

            # cross-conformal p-value: average per-fold p-values
            p_t = 0.0
            for clf, fold_cal in folds:
                s_t = clf.predict_proba(x)[0, 1]
                n_above = np.sum(fold_cal > s_t)
                n_tied = np.sum(fold_cal == s_t)
                n_ties += int(n_tied > 0)
                p_t += (n_above + rng.uniform() * (n_tied + 1)) / (len(fold_cal) + 1)
            p_t /= len(folds)

            # Simple Jumper update
            S_prev = C[-1] + C[0] + C[1]
            for eps in (-1, 0, 1):
                C[eps] = (1 - J) * C[eps] + (J / 3) * S_prev
            for eps in (-1, 0, 1):
                C[eps] *= 1 + eps * (p_t - 0.5)
            log_wealth_path.append(np.log(C[-1] + C[0] + C[1]))

        results.append({
            "run": labels["run"].iloc[ride_idx],
            "participant_id": labels["participant_id"].iloc[ride_idx],
            "n_ties": n_ties,
            "log_wealth_path": log_wealth_path,
        })

df_res = pd.DataFrame(results)
df_res

LOPO:   0%|                                                                                                                                                                      | 0/25 [00:00<?, ?it/s]

LOPO:   4%|██████▎                                                                                                                                                       | 1/25 [00:01<00:35,  1.49s/it]

LOPO:   8%|████████████▋                                                                                                                                                 | 2/25 [00:03<00:34,  1.50s/it]

LOPO:  12%|██████████████████▉                                                                                                                                           | 3/25 [00:04<00:35,  1.60s/it]

LOPO:  16%|█████████████████████████▎                                                                                                                                    | 4/25 [00:07<00:40,  1.91s/it]

...

LOPO:  92%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍            | 23/25 [00:44<00:03,  1.94s/it]

LOPO:  96%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋      | 24/25 [00:46<00:01,  1.98s/it]

LOPO: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25/25 [00:48<00:00,  1.88s/it]

LOPO: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25/25 [00:48<00:00,  1.92s/it]

,run,participant_id,n_ties,log_wealth_path
0,High,3.0,0,"[0.0, 0.13915896767268643, 0.3750075364083797,..."
1,Sober,3.0,0,"[0.0, 0.0687089129759368, 0.1858441937459419, ..."
2,High,3.0,0,"[0.0, 0.1451313240216315, 0.3837245091592695, ..."
3,Low,3.0,0,"[-1.1102230246251565e-16, 0.04028479120506093,..."
4,Low,3.0,0,"[0.0, 0.1298263135315167, 0.35230891349598853,..."
...,...,...,...,...
136,Sober,31.0,0,"[-1.1102230246251565e-16, 0.005731472584714349..."
137,Low,31.0,0,"[0.0, 0.12669747842839169, 0.3249331345102212,..."
138,Low,31.0,0,"[0.0, 0.003747591827538007, 0.0013628210307105..."
139,Sober,31.0,0,"[0.0, 0.005660841425905171, 0.0169280973158820..."


In [10]:
# Ville threshold: τ = log(1/α). By Ville's inequality, P(∃t : W_t ≥ 1/α) ≤ α under H0,
# so this fires with per-ride false-alarm probability ≤ α without any recalibration.
tau = np.log(1.0 / alpha)
detection_step = []

for path in df_res["log_wealth_path"]:
    alarm_steps = np.flatnonzero(np.asarray(path) > tau)
    detection_step.append(int(alarm_steps[0]) if len(alarm_steps) else None)

df_res["detection_step"] = detection_step
df_res["detected"] = [s is not None for s in detection_step]
df_res["detection_time_s"] = [
    (s * pe_s + pe_w) / sampling_rate if s is not None else None for s in detection_step
]
df_res

,run,participant_id,n_ties,log_wealth_path,detection_step,detected,detection_time_s
0,High,3.0,0,"[0.0, 0.13915896767268643, 0.3750075364083797,...",12.0,True,22.0
1,Sober,3.0,0,"[0.0, 0.0687089129759368, 0.1858441937459419, ...",21.0,True,31.0
2,High,3.0,0,"[0.0, 0.1451313240216315, 0.3837245091592695, ...",13.0,True,23.0
3,Low,3.0,0,"[-1.1102230246251565e-16, 0.04028479120506093,...",NaN,False,NaN
4,Low,3.0,0,"[0.0, 0.1298263135315167, 0.35230891349598853,...",14.0,True,24.0
...,...,...,...,...,...,...,...
136,Sober,31.0,0,"[-1.1102230246251565e-16, 0.005731472584714349...",NaN,False,NaN
137,Low,31.0,0,"[0.0, 0.12669747842839169, 0.3249331345102212,...",13.0,True,23.0
138,Low,31.0,0,"[0.0, 0.003747591827538007, 0.0013628210307105...",20.0,True,30.0
139,Sober,31.0,0,"[0.0, 0.005660841425905171, 0.0169280973158820...",NaN,False,NaN


In [11]:
_ = bootstrap_report(df_res, glue=glue, per_ride_path=per_ride_path)

       support     detection_rate det_time_min det_time_q25 det_time_med det_time_q75 det_time_max                FAR
run                                                                                                                  
High        46  1.00 [1.00, 1.00]  22 [22, 22]  22 [22, 22]  23 [22, 23]  24 [23, 25]  36 [26, 36]                NaN
Low         50  0.94 [0.86, 1.00]  22 [22, 22]  23 [22, 24]  25 [24, 26]  30 [27, 33]  48 [35, 48]                NaN
Sober       45                NaN          NaN          NaN          NaN          NaN          NaN  0.67 [0.52, 0.80]

accuracy sober vs non-sober: 0.77 [0.72, 0.82]
accuracy sober vs high:      0.67 [0.60, 0.74]


/home/marco/git/online-impairment-detection/.venv/lib/python3.13/site-packages/jupyter_client/session.py:727: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


In [12]:
# did smoothing matter?
total_ties = int(df_res["n_ties"].sum())
rides_with_ties = int((df_res["n_ties"] > 0).sum())
print(f"calibration-score ties: {total_ties} across {rides_with_ties}/{len(df_res)} rides")

calibration-score ties: 0 across 0/141 rides
